이전 1작기의 정면 bed-seg 모델로 2작기의 데이터를 잘라내려 했으나, 인식에 완전히 실패하였다.
그래서 260409일에 550장을 다시 라벨링하였고, 여기 하단에서는 1작기 bed-seg 모델에 2작기 데이터를 fine-tuning 한것을 해보도록 할 예정이다

In [ ]:
!pip -q install ultralytics opencv-python pandas tqdm openpyxl

In [ ]:
# bed-seg (2작기 TEST) - content 복사 없이 Drive 경로에서 바로 추론
# =========================================================
# 요구사항 반영
# 1) 2790장: Drive -> /content 복사 생략(필요시 옵션으로만)
# 2) Colab Pro 용량/메모리: chunk + 주기적 gc/empty_cache 유지
# 3) 병렬: (a) 파일 스캔/저장 병렬, (b) 추론은 GPU 안정 위해 단일 스트림 권장
# 4) 베드 끝/기준점(원) 잘림 보완: bbox padding(비율+최소px) + 경계접촉시 추가 패딩
# =========================================================

# [Colab 권장] 런타임 시작 후 먼저 실행
# !pip -q install ultralytics opencv-python pandas tqdm openpyxl

import os
import gc
import json
import time
import math
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO

# -----------------------------
# ✅ 사용자 설정 (여기만 수정)
# -----------------------------
INPUT_ROOT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/2작기/260306-260402/260308"

OUT_ROOT = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/260306-260402/260308"

MODEL_PATH = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/1작기/results/runs/bed_seg_v1/weights/best.pt"

# 이미지 확장자
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# 추론 파라미터
CONF = 0.001
IOU = 0.7
IMGSZ = 832
BATCH = 2
DEVICE = 'cpu'  # GPU면 0, CPU면 'cpu'
MAX_DET = 3
USE_HALF = True

# chunk/메모리
CHUNK_SIZE = 400
CLEAN_EVERY_N = 50
RESUME = True

# 저장 옵션
SAVE_BED_CROPS = True           # ✅ bbox padding 적용된 bed crop 이미지 저장
SAVE_BED_CROPS_MAX_PER_IMG = 2  # 이미지당 저장할 bed crop 개수(상위 score 순)

# 병렬 저장(이미지 write는 병렬해도 안정적)
N_WORKERS_SAVE = min(16, (os.cpu_count() or 8))


In [ ]:
# -----------------------------
# ✅ 잘림 보완: bbox padding 파라미터
# -----------------------------
PAD_X_RATIO = 0.12
PAD_Y_RATIO = 0.15
MIN_PAD_PX = 24

BORDER_EPS_PX = 6
EXTRA_PAD_BORDER_PX = 60

# -----------------------------
# 유틸
# -----------------------------

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def list_images_recursive(root_dir: str, exts=None):
    if exts is None:
        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    root = Path(root_dir)
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            files.append(str(p))
    return sorted(files)

def safe_relpath(path: str, start: str):
    try:
        return os.path.relpath(path, start)
    except Exception:
        return Path(path).name

def _chunks(lst, chunk_size: int):
    for i in range(0, len(lst), chunk_size):
        yield i // chunk_size, lst[i:i + chunk_size]

def polygon_area(poly_xy: np.ndarray) -> float:
    if poly_xy is None or len(poly_xy) < 3: return 0.0
    x, y = poly_xy[:, 0], poly_xy[:, 1]
    return float(0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))

def masks_to_polygons(result):
    if getattr(result, "masks", None) is None: return []
    polys = result.masks.xy
    return polys if polys is not None else []

def compute_padded_bbox(xyxy, w, h, pad_x_ratio=PAD_X_RATIO, pad_y_ratio=PAD_Y_RATIO, min_pad_px=MIN_PAD_PX, border_eps_px=BORDER_EPS_PX, extra_pad_border_px=EXTRA_PAD_BORDER_PX):
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    bw, bh = max(0.0, x2 - x1), max(0.0, y2 - y1)
    pad_x, pad_y = max(min_pad_px, bw * pad_x_ratio), max(min_pad_px, bh * pad_y_ratio)
    touch_left, touch_top = x1 <= border_eps_px, y1 <= border_eps_px
    touch_right, touch_bottom = x2 >= (w - border_eps_px), y2 >= (h - border_eps_px)
    pad_l = pad_x + (extra_pad_border_px if touch_left else 0)
    pad_t = pad_y + (extra_pad_border_px if touch_top else 0)
    pad_r = pad_x + (extra_pad_border_px if touch_right else 0)
    pad_b = pad_y + (extra_pad_border_px if touch_bottom else 0)
    nx1, ny1 = max(0, int(math.floor(x1 - pad_l))), max(0, int(math.floor(y1 - pad_t)))
    nx2, ny2 = min(w - 1, int(math.ceil(x2 + pad_r))), min(h - 1, int(math.ceil(y2 + pad_b)))
    return (nx1, ny1, nx2, ny2), {"touch_left": bool(touch_left), "touch_top": bool(touch_top), "touch_right": bool(touch_right), "touch_bottom": bool(touch_bottom)}

def save_crop_one(img_path, crop_xyxy, out_path):
    try:
        img = cv2.imread(img_path)
        if img is None: return False, f"imread fail: {img_path}"
        x1, y1, x2, y2 = crop_xyxy
        crop = img[y1:y2, x1:x2]
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        ok = cv2.imwrite(out_path, crop)
        return ok, None if ok else f"imwrite fail: {out_path}"
    except Exception as e: return False, str(e)

def process_single_image(result, input_root, crop_dir):
    img_path = str(result.path)
    rel = safe_relpath(img_path, input_root)
    orig = cv2.imread(img_path)
    if orig is None: return None, {"image_rel": rel, "image_abs": img_path, "n_instances": 0, "read_fail": 1}
    H, W = orig.shape[:2]
    polys = masks_to_polygons(result)
    boxes_xyxy = result.boxes.xyxy.cpu().numpy() if (result.boxes is not None and len(result.boxes) > 0) else np.zeros((0, 4))
    confs = result.boxes.conf.cpu().numpy() if (result.boxes is not None and len(result.boxes) > 0) else np.zeros((0,))
    K = max(len(polys), len(confs))
    best_idx, best_score, rows, save_jobs = None, -1.0, [], []
    inst_meta = []
    for i in range(K):
        poly = polys[i] if i < len(polys) else None
        conf = float(confs[i]) if i < len(confs) else float("nan")
        box = boxes_xyxy[i].tolist() if i < len(boxes_xyxy) else [float("nan")]*4
        area = polygon_area(np.array(poly)) if poly is not None else (max(0, box[2]-box[0])*max(0, box[3]-box[1]))
        score = (0.0 if math.isnan(conf) else conf) * area
        if score > best_score: best_score, best_idx = score, i
        pad_xyxy, pad_info = compute_padded_bbox(box, W, H) if not any(math.isnan(v) for v in box) else (None, None)
        inst_meta.append((i, score, pad_xyxy, pad_info, box, conf, area, json.dumps(poly.tolist()) if poly is not None else ""))
    inst_meta_sorted = sorted(inst_meta, key=lambda x: x[1], reverse=True)[:SAVE_BED_CROPS_MAX_PER_IMG]
    for (i, score, pad_xyxy, pad_info, box, conf, area, poly_json) in inst_meta:
        crop_rel = ""
        if SAVE_BED_CROPS and pad_xyxy and any(i == s[0] for s in inst_meta_sorted):
            crop_rel = str(Path(rel).with_suffix("")) + f"__bed{i:02d}.jpg"
            save_jobs.append((img_path, pad_xyxy, os.path.join(crop_dir, crop_rel)))
        rows.append({"image_rel": rel, "W": W, "H": H, "inst_idx": i, "conf": conf, "area_px": area, "score_conf_area": score, "bbox_pad_xyxy": json.dumps(pad_xyxy), "crop_rel": crop_rel})
    return (rows, {"image_rel": rel, "n_instances": K, "best_score_conf_area": best_score, "read_fail": 0}, save_jobs)

def run_bedseg_chunked(input_root=INPUT_ROOT, out_root=OUT_ROOT, chunk_size=CHUNK_SIZE, resume=RESUME):
    ensure_dir(out_root); chunk_dir = os.path.join(out_root, "chunks"); ensure_dir(chunk_dir)
    crop_dir = os.path.join(out_root, "bed_crops_pad"); ensure_dir(crop_dir)
    img_files = list_images_recursive(input_root)
    model = YOLO(MODEL_PATH)
    overall = tqdm(total=len(img_files), desc="[Parallel] bed-seg")
    for chunk_idx, chunk_files in _chunks(img_files, chunk_size):
        tag = f"chunk_{chunk_idx:04d}"
        out_sum, out_long = os.path.join(chunk_dir, f"{tag}_sum.csv"), os.path.join(chunk_dir, f"{tag}_long.csv")
        if resume and os.path.exists(out_sum): overall.update(len(chunk_files)); continue
        results = model.predict(source=chunk_files, conf=CONF, iou=IOU, imgsz=IMGSZ, device=DEVICE, batch=BATCH, stream=True, verbose=False)
        all_rows, all_sum, all_save = [], [], []
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(process_single_image, res, input_root, crop_dir) for res in results]
            for f in as_completed(futures):
                r, s, j = f.result()
                if r: all_rows.extend(r)
                all_sum.append(s)
                all_save.extend(j)
                overall.update(1)
        with ThreadPoolExecutor(max_workers=N_WORKERS_SAVE) as ex:
            [ex.submit(save_crop_one, *job) for job in all_save]
        pd.DataFrame(all_rows).to_csv(out_long, index=False); pd.DataFrame(all_sum).to_csv(out_sum, index=False)
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    overall.close()
    return "Done"

In [ ]:
# =============================
# 실행부 (병렬 처리 버전)
# =============================
if __name__ == "__main__":
    # 4개 스레드를 활용한 병렬 추론 및 저장을 시작합니다.
    info = run_bedseg_chunked()
    print(f"최종 결과: {info}")

[Parallel] bed-seg:   0%|          | 0/89 [00:00<?, ?it/s]

#89장 한정

In [ ]:
import os
import time
import pandas as pd
from ultralytics import YOLO
from datetime import datetime

# ================================================================= #
# 1. 경로 및 설정 (사용자 수정 구역)
# ================================================================= #
# 원본 이미지 경로 (260306 등의 날짜 하위 폴더가 있는 곳)
IMG_ROOT  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/2작기/260306-260402/260308"

SAVE_ROOT  = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/260306-260402/260308"

MODEL_PATH = r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/1작기/results/runs/bed_seg_v1/weights/best.pt"

# 배치 사이즈 (GPU 메모리에 따라 8~16 권장)
BATCH_SIZE = 16

# ================================================================= #
# 2. 실행 함수
# ================================================================= #

def run_bed_segmentation():
    start_all = time.time()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 작업을 시작합니다.")

    # 1. 모델 딱 한 번만 로드
    print("모델 로딩 중...")
    model = YOLO(MODEL_PATH)

    # 2. 처리할 이미지 리스트 전체 확보 (하위 폴더 포함)
    image_paths = []
    for root, dirs, files in os.walk(IMG_ROOT):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(root, f))

    total_count = len(image_paths)
    print(f"총 {total_count}개의 이미지를 발견했습니다.")

    # 3. 배치 단위로 끊어서 처리
    results_summary = []

    for i in range(0, total_count, BATCH_SIZE):
        batch_start = time.time()
        batch_files = image_paths[i : i + BATCH_SIZE]

        # YOLO 배치 추론 (전역 변수 DEVICE 사용)
        results = model.predict(source=batch_files, conf=0.25, device=DEVICE, save=False)

        for j, r in enumerate(results):
            original_path = batch_files[j]
            filename = os.path.basename(original_path)

            # 날짜 폴더명 추출
            rel_path = os.path.relpath(original_path, IMG_ROOT)
            sub_folder = os.path.dirname(rel_path)

            # 저장 경로 설정 및 생성
            save_dir = os.path.join(SAVE_ROOT, sub_folder)
            os.makedirs(save_dir, exist_ok=True)

            # 결과 이미지 저장
            r.save(filename=os.path.join(save_dir, f"seg_{filename}"))

            # 데이터 요약 정보
            results_summary.append({
                'date_folder': sub_folder,
                'filename': filename,
                'boxes': len(r.boxes),
                'status': 'Success'
            })

        batch_end = time.time()
        print(f"Progress: [{min(i + BATCH_SIZE, total_count)}/{total_count}] - 소요시간: {batch_end - batch_start:.2f}초")

    # 4. 결과 리포트 저장
    if results_summary:
        df = pd.DataFrame(results_summary)
        csv_path = os.path.join(SAVE_ROOT, f"process_log_{datetime.now().strftime('%y%m%d_%H%M')}.csv")
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"로그 저장 완료: {csv_path}")

    end_all = time.time()
    print("\n" + "="*50)
    print(f"전체 작업 완료!")
    print(f"총 소요 시간: {end_all - start_all:.2f}초 (약 {(end_all - start_all)/60:.2f}분)")
    print("="*50)

In [ ]:

# ================================================================= #
# 3. 실행부
# ================================================================= #
if __name__ == "__main__":
    run_bed_segmentation()

[16:07:27] 작업을 시작합니다.
모델 로딩 중...
총 89개의 이미지를 발견했습니다.

0: 736x960 (no detections), 824.5ms
1: 736x960 (no detections), 824.5ms
2: 736x960 (no detections), 824.5ms
3: 736x960 (no detections), 824.5ms
4: 736x960 (no detections), 824.5ms
5: 736x960 (no detections), 824.5ms
6: 736x960 (no detections), 824.5ms
7: 736x960 (no detections), 824.5ms
8: 736x960 (no detections), 824.5ms
9: 736x960 (no detections), 824.5ms
10: 736x960 (no detections), 824.5ms
11: 736x960 (no detections), 824.5ms
12: 736x960 (no detections), 824.5ms
13: 736x960 (no detections), 824.5ms
14: 736x960 (no detections), 824.5ms
15: 736x960 (no detections), 824.5ms
Speed: 11.9ms preprocess, 824.5ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 960)
Progress: [16/89] - 소요시간: 17.14초

0: 736x960 (no detections), 772.7ms
1: 736x960 (no detections), 772.7ms
2: 736x960 (no detections), 772.7ms
3: 736x960 (no detections), 772.7ms
4: 736x960 (no detections), 772.7ms
5: 736x960 (no detections), 772.7ms
6: 736x960 (no 